# Viscosity Interpolation

This notebook is the copmutation and generation of plots for the REport:

Viscosity Variation with Temperature

Draft in Google Drive here:

https://docs.google.com/document/d/1rMQxd-Eqf6_fCpH0JU7dY3RrDwhEmaOF/edit?usp=sharing&ouid=112738556987743288660&rtpof=true&sd=true

This notebook, when run, should generate the plots for that document.


In [6]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot  as plt
import matplotlib
import adios_db.scripting as ads
import adios_db.computation.physical_properties as pp
import csv
import scipy

In [7]:
# Set the data dir for your machine
DATA_PATHS = [Path(r"C:\Users\rintaro.moriyasu\Oil Databases\noaa-oil-data\data\oil"),
              Path(r"/Users/chris.barker/Hazmat/GitLab/noaa-oil-data/data/oil"),
             ]
# Look for which one is real
for p in DATA_PATHS:
    if p.is_dir():
        DATA_DIR = p
        break
print("Using data in:", DATA_DIR)

Using data in: /Users/chris.barker/Hazmat/GitLab/noaa-oil-data/data/oil


In [25]:
# these are orimulsions
no_list = ['AD01706', 'EC00662', 'AD02290', 'AD02599', 'AD00005', 'AD00458', 'AD00459'] 
product_types_to_use = {
"Crude Oil NOS",
"Condensate",
# "Bitumen Blend",
# "Bitumen",
# "Refined Product NOS",
# "Distillate Fuel Oil",
# "Residual Fuel Oil",
# "Refinery Intermediate",
# "Solvent",
# "Bio-fuel Oil",
# "Natural Plant Oil",
# "Lube Oil",
# "Dielectric Oil",
# "Other"
}

oil_records = {}

# get all the records, and sort by product type
for oil, filename in ads.get_all_records(DATA_DIR):
    product_type = oil.metadata.product_type
    if oil.oil_id in no_list:
        continue
    if product_type in product_types_to_use:
        oil_records.setdefault(product_type, []).append(oil)


product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Refinery Intermediate
returning Refinery Intermediate
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oi

product type in: Bitumen Blend
returning Bitumen Blend
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Bitumen Blend
returning Bitumen Blend
product type in: Bio-fuel Oil
returning Bio-fuel Oil
product type in: Bio-Petro Fuel Oil
returning Bio-Petro Fuel Oil
product type in: Bio-Petro Fuel Oil
returning Bio-Petro Fuel Oil
product type in: Bio-fuel Oil
returning Bio-fuel Oil
product type in: Bio-Petro Fuel Oil
returning Bio-Petro Fuel Oil
product type in: Bio-Petro Fuel Oil
returning Bio-Petro Fuel Oil
product type in: Bio-fuel Oil
returning Bio-fuel Oil
product type in: Bio-Petro Fuel Oil
returning Bio-Petro Fuel Oil
product type in: Bio-Petro Fuel Oil
returning Bio-Petro Fuel Oil
product type in: Bio-fuel Oil
returning Bio-fuel Oil
product type in: Bio-Petro Fuel Oil
returning Bio-Petro Fuel Oil
product type in: Bio-Petro Fuel Oil
returning Bio-Petro Fu

product type in: Condensate
returning Condensate
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in: Crude Oil NOS
returning Crude Oil NOS
product type in:

In [26]:
print("Got data for: ", oil_records.keys())

Got data for:  dict_keys(['Crude Oil NOS', 'Condensate'])


In [33]:
# Do the calculation for each product type:
for pt, oil_rec in oil_records.items():
    oil_recs = oil_records[pt]
    # find the records that have more than two Kinematic Viscosities
    kv2_values = []
    density_values = []
    oil_ids = []
    for oil in oil_recs:
        kv_data = oil.sub_samples[0].physical_properties.kinematic_viscosities
        if not kv_data: # if no kinematic vicosity, use dynamic
            kv_data = oil.sub_samples[0].physical_properties.dynamic_viscosities
            if not kv_data:
                # no dynamic vicosity data either -- skip
                continue
        errors = kv_data.validate()
        if errors: # somethign wrong with the data -- don't use.
            continue
        try:
            KV = pp.KinematicViscosity(oil)
        except ValueError: # couldn't create a KinematicViscosity object -- why not?
             raise # continue
        except:
            print(oil.oil_id)
            print(oil.sub_samples[0].physical_properties.kinematic_viscosities)
            print(oil.sub_samples[0].physical_properties.dynamic_viscosities)
            print(oil.sub_samples[0].physical_properties.densities)
            raise
        # make sure there is density data, too:
        if len(oil.sub_samples[0].physical_properties.densities) == 0:
            continue          
        if len(KV.kviscs) > 1:
            k_v2 = KV._k_v2
            #if k_v2 < 28000:
            density = pp.Density(oil)
            d = density.at_temp(15, unit = 'C')
            if True:
                density_values.append(d)
                #print(density_values[-1], oil.oil_id)
                kv2_values.append(k_v2)
                oil_ids.append(oil.oil_id)
    #         else:
    #             print("Removing", oil.oil_id)
    #         if k_v2 > 24000:
    #             print(oil.oil_id)
    fig, ax = plt.subplots()
    ax.hist(kv2_values, bins = 20)
    ax.set_title("Coefficient of Kinematic Viscosities ($k_v$): " + pt)
    plt.xlabel("Coefficient of Kinematic Viscosities")
    plt.ylabel("Number of Oils")
    
    fig.savefig(f"kvisc_plot_{pt}.png", dpi=600)
    fig, ax = plt.subplots()
    ax.scatter(density_values, kv2_values)
    ax.set_title("Density/Coefficient of Kinematic Viscosity ($k_v$): " + pt)
    plt.xlabel("Densities")
    plt.ylabel("Coeff. of Kinematic Viscosities")
    density_values = np.array(density_values)
    res = scipy.stats.linregress(density_values, kv2_values, alternative='two-sided')
    plt.plot(density_values, res.intercept + res.slope*density_values, 'r', label='fitted line')
    fig.savefig(f"densitykvis_plot_{pt}.png", dpi=600)

AD02069
KinematicViscosityList([])
DynamicViscosityList([DynamicViscosityPoint(viscosity=DynamicViscosity(unit='mPas', unit_type='dynamicviscosity'), ref_temp=Temperature(value=0.0, unit='C', unit_type='temperature')), DynamicViscosityPoint(viscosity=DynamicViscosity(value=235000.0, unit='mPas', unit_type='dynamicviscosity'), ref_temp=Temperature(value=15.0, unit='C', unit_type='temperature'))])
DensityList([DensityPoint(density=Density(value=1.0075, unit='g/mL', unit_type='density'), ref_temp=Temperature(value=0.0, unit='C', unit_type='temperature')), DensityPoint(density=Density(value=1.0049, unit='g/mL', unit_type='density'), ref_temp=Temperature(value=5.0, unit='C', unit_type='temperature')), DensityPoint(density=Density(value=1.0023, unit='g/mL', unit_type='density'), ref_temp=Temperature(value=10.0, unit='C', unit_type='temperature')), DensityPoint(density=Density(value=1.0002, unit='g/mL', unit_type='density'), ref_temp=Temperature(value=15.0, unit='C', unit_type='temperature'))

TypeError: unsupported operand type(s) for /: 'NoneType' and 'float'

In [23]:
# mean_kv2 = np.average(kv2_values)
# print("Mean is", mean_kv2)
# stdev_kv2 = np.std(kv2_values)
# print("Standard Deviation is", stdev_kv2)
# median_kv2 = np.median(kv2_values)
# print("Median is", median_kv2)
# print("Max Value is", max(kv2_values))

In [ ]:
residuals = {}
no_list = ['AD00813', 'NO00119', 'NO00120','NO00121','NO00122', 'AD02582', 'AD01485' ]
matplotlib.rcParams['figure.max_open_warning'] = 240
for oil in oils[:]:
    validation = oil.validate()
    bad_oil = False
    for message in validation:
        if message.startswith("E062"):
            print("Skipping", oil.oil_id)
            bad_oil = True
            break
    if oil.oil_id in no_list:
        continue
    if bad_oil:
        continue
    
        
    oil_name = oil.metadata.name
    #print(oil_name)
    KV = pp.KinematicViscosity(oil)

    visc = KV.kviscs
    if len(visc) < 3:
        continue
    if KV.residuals[0] < 0.07:
        continue
    temps = KV.temps
    visc2 = KV.at_temp(temps)
    residuals[oil.oil_id] = KV.residuals[0]
    temps3 = np.linspace(min(temps), max(temps))
    visc3 = KV.at_temp(temps3)
    fig, ax = plt.subplots()
    ax.set_title(oil_name +" " + oil.oil_id)
    ax.plot(temps, visc, 'o')
    ax.plot(temps, visc2, 'x')
    ax.plot(temps3, visc3)
    ax.legend(["original points", "re-computed", "interpolator"])
    ax.text(min(temps3),min(visc3) , f"Residual: {KV.residuals}")


In [ ]:
res = list(residuals.values())
fig, ax = plt.subplots()
ax.hist(res, bins = 20)
#     ax.set_title(oil_name +" " + oil.oil_id)
#     ax.plot(temps, visc, 'o')
#     ax.plot(temps, visc2, 'x')
#     ax.plot(temps3, visc3)
#     ax.legend(["original points", "re-computed", "interpolator"])
#     ax.text(min(temps3),min(visc3) , f"Residual: {KV.residuals}")

In [ ]:
max(res)
res_reversed = {res:id for id, res in residuals.items()}
res_reversed[max(res)]
res_reversed[min(res)]

In [ ]:
matplotlib.rcParams['figure.max_open_warning'] = 240

In [ ]:
oil_ids
min(kv2_values)
np.argmin(kv2_values)
oil_ids[46]